# Experiment

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, sqrt, lit

# Create a Spark session
spark = SparkSession.builder.appName("weighted_summary").getOrCreate()

# Assume df is your DataFrame with columns: 'groupid', 'x', 'y', and 'weight'

# Calculate the weighted mean
df_with_weighted_mean = df.withColumn("x_weighted", col("x") * col("weight")) \
                          .withColumn("y_weighted", col("y") * col("weight"))

df_grouped = df_with_weighted_mean.groupBy("groupid").agg(_sum("x_weighted").alias("sum_x_weighted"), 
                                                           _sum("y_weighted").alias("sum_y_weighted"), 
                                                           _sum("weight").alias("sum_weight"))

df_grouped = df_grouped.withColumn("weighted_mean_x", col("sum_x_weighted") / col("sum_weight")) \
                       .withColumn("weighted_mean_y", col("sum_y_weighted") / col("sum_weight"))

# Add the weighted mean to the original dataframe
df_with_weighted_mean = df_with_weighted_mean.join(df_grouped.select("groupid", "weighted_mean_x", "weighted_mean_y"), on="groupid", how="left")

# Calculate the weighted standard deviation
df_with_weighted_sd = df_with_weighted_mean.withColumn("x_diff_squared_weighted", ((col("x") - col("weighted_mean_x"))**2) * col("weight")) \
                                            .withColumn("y_diff_squared_weighted", ((col("y") - col("weighted_mean_y"))**2) * col("weight"))

df_grouped_sd = df_with_weighted_sd.groupBy("groupid").agg(_sum("x_diff_squared_weighted").alias("sum_x_diff_squared_weighted"), 
                                                           _sum("y_diff_squared_weighted").alias("sum_y_diff_squared_weighted"),
                                                           _sum("weight").alias("sum_weight"))

df_grouped_sd = df_grouped_sd.withColumn("weighted_sd_x", sqrt(col("sum_x_diff_squared_weighted") / col("sum_weight"))) \
                             .withColumn("weighted_sd_y", sqrt(col("sum_y_diff_squared_weighted") / col("sum_weight")))

# Display the final result
df_grouped_sd.show()
